In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shubhamgovekar/deberta-training/deberta_training_50k_cleaned.csv


In [2]:
# Pin to 4.38.2 — 4.40.0 breaks TFDebertaV2 .compile() on Kaggle's
# TF 2.16 / tf-keras stack (_distribute_strategy AttributeError).
!# Pin to 4.38.2 — 4.40.0 breaks TFDebertaV2 .compile()
!pip install -q "transformers==4.38.2" "sentencepiece" "protobuf"

# 1. SET THIS BEFORE ANY ML IMPORTS
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# 2. Now you can safely import the rest of your stack
import math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf

# ... the rest of your code remains exactly the same ...
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, create_optimizer

# float32 — no mixed precision, avoids LossScaler NaN at lr=2e-5
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")

# GPU memory growth
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {len(tf.config.list_physical_devices('GPU'))}")
print(f"Precision  : {mixed_precision.global_policy().name}")
print("Setup complete ✓")

2026-04-04 08:49:03.370477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775292543.393770     772 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775292543.401172     772 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775292543.419091     772 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775292543.419110     772 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775292543.419112     772 computation_placer.cc:177] computation placer alr

TensorFlow : 2.19.0
GPUs found : 2
Precision  : float32
Setup complete ✓


In [3]:
class CFG:
    # ── paths ─────────────────────────────────────────────────────────────────
    DATA_PATH        = "/kaggle/input/datasets/shubhamgovekar/deberta-training/deberta_training_50k_cleaned.csv"
    MODEL_NAME       = "microsoft/deberta-v3-small"
    OUTPUT_DIR       = "/kaggle/working/backend/model"
    CHECKPOINT_PATH  = "/kaggle/working/checkpoints/best_model.weights.h5"

    # ── data ──────────────────────────────────────────────────────────────────
    MAX_LEN          = 128
    VAL_SPLIT        = 0.10
    TEST_SPLIT       = 0.10
    RANDOM_SEED      = 42

    # ── training ──────────────────────────────────────────────────────────────
    # Reduced from 32 → 16: DeBERTa-v3-small + float32 is heavier than it looks.
    # 16 keeps you safely under T4's 16 GB. Effective LR unchanged.
    BATCH_SIZE       = 16
    EPOCHS           = 5
    LEARNING_RATE    = 2e-5
    WEIGHT_DECAY     = 0.01
    WARMUP_RATIO     = 0.10
    DROPOUT          = 0.1

    # ── early stopping ────────────────────────────────────────────────────────
    ES_PATIENCE      = 2
    ES_MIN_DELTA     = 1e-4

    # ── labels ────────────────────────────────────────────────────────────────
    LABEL2ID         = {"Genuine": 0, "Fake": 1}
    ID2LABEL         = {0: "Genuine", 1: "Fake"}
    NUM_LABELS       = 2

cfg = CFG()
print("Config loaded ✓")
print(f"  Data path  : {cfg.DATA_PATH}")
print(f"  Batch size : {cfg.BATCH_SIZE}")
print(f"  Max len    : {cfg.MAX_LEN}")


Config loaded ✓
  Data path  : /kaggle/input/datasets/shubhamgovekar/deberta-training/deberta_training_50k_cleaned.csv
  Batch size : 16
  Max len    : 128


In [4]:
def load_and_split(cfg):
    print("\n[1/4] Loading dataset …")
    df = pd.read_csv(cfg.DATA_PATH)

    assert "review_text" in df.columns and "label" in df.columns, \
        "CSV must have 'review_text' and 'label' columns."

    df = df[["review_text", "label"]].dropna()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df["label"]       = df["label"].astype(int)

    print(f"  Total  : {len(df):,} rows")
    print(f"  Labels : {df['label'].value_counts().to_dict()}")

    df_train, df_temp = train_test_split(
        df, test_size=cfg.VAL_SPLIT + cfg.TEST_SPLIT,
        stratify=df["label"], random_state=cfg.RANDOM_SEED
    )
    val_ratio = cfg.VAL_SPLIT / (cfg.VAL_SPLIT + cfg.TEST_SPLIT)
    df_val, df_test = train_test_split(
        df_temp, test_size=1 - val_ratio,
        stratify=df_temp["label"], random_state=cfg.RANDOM_SEED
    )
    print(f"  Train:{len(df_train):,}  Val:{len(df_val):,}  Test:{len(df_test):,}")
    return df_train, df_val, df_test


def tokenize_df(df, tokenizer, cfg):
    enc = tokenizer(
        df["review_text"].tolist(),
        max_length=cfg.MAX_LEN, padding="max_length",
        truncation=True, return_tensors="np"
    )
    return {
        "input_ids"      : enc["input_ids"],
        "attention_mask" : enc["attention_mask"],
        "token_type_ids" : enc.get("token_type_ids", np.zeros_like(enc["input_ids"])),
        "labels"         : df["label"].values,
    }


def to_tf_dataset(enc, cfg, shuffle=False):
    feats = {k: v for k, v in enc.items() if k != "labels"}
    ds = tf.data.Dataset.from_tensor_slices((feats, enc["labels"]))
    if shuffle:
        ds = ds.shuffle(len(enc["labels"]), seed=42, reshuffle_each_iteration=True)
    return ds.batch(cfg.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


print("\n[2/4] Tokenising …")
df_train, df_val, df_test = load_and_split(cfg)

tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)

train_ds = to_tf_dataset(tokenize_df(df_train, tokenizer, cfg), cfg, shuffle=True)
val_ds   = to_tf_dataset(tokenize_df(df_val,   tokenizer, cfg), cfg)
test_ds  = to_tf_dataset(tokenize_df(df_test,  tokenizer, cfg), cfg)

print("Tokenisation complete ✓")
print(f"  Train batches : {len(train_ds)}")
print(f"  Val batches   : {len(val_ds)}")
print(f"  Test batches  : {len(test_ds)}")


[2/4] Tokenising …

[1/4] Loading dataset …
  Total  : 50,000 rows
  Labels : {0: 25000, 1: 25000}
  Train:40,000  Val:5,000  Test:5,000


I0000 00:00:1775292577.392970     772 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775292577.398123     772 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Tokenisation complete ✓
  Train batches : 2500
  Val batches   : 313
  Test batches  : 313


In [5]:
print("\n[3/4] Building model …")

model = TFAutoModelForSequenceClassification.from_pretrained(
    cfg.MODEL_NAME,
    num_labels                   = cfg.NUM_LABELS,
    id2label                     = cfg.ID2LABEL,
    label2id                     = cfg.LABEL2ID,
    hidden_dropout_prob          = cfg.DROPOUT,
    attention_probs_dropout_prob = cfg.DROPOUT,
)

# ── Scheduler & optimiser ────────────────────────────────────────────────────
num_train_steps  = math.ceil(len(train_ds) * cfg.EPOCHS)
num_warmup_steps = int(num_train_steps * cfg.WARMUP_RATIO)

optimizer, schedule = create_optimizer(
    init_lr           = cfg.LEARNING_RATE,
    num_train_steps   = num_train_steps,
    num_warmup_steps  = num_warmup_steps,
    weight_decay_rate = cfg.WEIGHT_DECAY,
)

print(f"  Train steps  : {num_train_steps:,}")
print(f"  Warmup steps : {num_warmup_steps:,}")

# ── Compile ──────────────────────────────────────────────────────────────────
# The HuggingFace TF model computes loss internally when 'labels' are in the
# input dict (loss=None). This is correct — keep it as-is.
# SparseCategoricalAccuracy tracks accuracy from logits correctly.
model.compile(
    optimizer = optimizer,
    metrics   = [tf.keras.metrics.SparseCategoricalAccuracy("accuracy")],
)

# ── Callbacks ────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(cfg.CHECKPOINT_PATH), exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath          = cfg.CHECKPOINT_PATH,
        monitor           = "val_loss",
        save_best_only    = True,
        save_weights_only = True,
        verbose           = 1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor              = "val_loss",
        patience             = cfg.ES_PATIENCE,
        min_delta            = cfg.ES_MIN_DELTA,
        restore_best_weights = True,
        verbose              = 1,
    ),
    tf.keras.callbacks.TerminateOnNaN(),
   # ── LR logger ────────────────────────────────────────────────────────────
    tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f"  LR after epoch {epoch+1}: {optimizer.learning_rate(optimizer.iterations).numpy():.2e}"
        )
    ),
]

# ── Train ─────────────────────────────────────────────────────────────────────
print("\n[4/4] Training …")
history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = cfg.EPOCHS,
    callbacks       = callbacks,
    verbose         = 1,
)
print("Training complete ✓")


[3/4] Building model …


All model checkpoint layers were used when initializing TFDebertaV2ForSequenceClassification.

Some layers of TFDebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier', 'pooler', 'cls_dropout']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Train steps  : 12,500
  Warmup steps : 1,250

[4/4] Training …
Epoch 1/5
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
2500/2500 [==============================] - ETA: 0s - loss: 0.1649 - accuracy: 0.9190
Epoch 1: val_loss improved from inf to 0.05295, saving model to /kaggle/working/checkpoints/best_model.weights.h5
  LR after epoch 1: 1.78e-05
2500/2500 [==============================] - 1018s 399ms/step - loss: 0.1649 - accuracy: 0.9190 - val_loss: 0.0530 - val_accuracy: 0.9810
Epoch 2/5
2500/2500 [==============================] - ETA: 0s - loss: 0.0440 - accur

In [6]:
def evaluate(model, test_ds, history, cfg):
    print("\nRunning inference on test set …")

    all_logits, all_labels = [], []
    for batch_feats, batch_labels in test_ds:
        out = model(batch_feats, training=False)
        all_logits.append(out.logits.numpy())
        all_labels.append(batch_labels.numpy())

    logits = np.concatenate(all_logits)
    y_true = np.concatenate(all_labels)
    y_pred = np.argmax(logits, axis=-1)

    target_names = [cfg.ID2LABEL[i] for i in range(cfg.NUM_LABELS)]
    acc = accuracy_score(y_true, y_pred)

    print("\n" + "="*60)
    print("  TEST SET RESULTS")
    print("="*60)
    print(f"  Accuracy : {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — Test Set", fontweight="bold")
    plt.tight_layout()
    plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
    plt.show()
    print("Confusion matrix saved ✓")

    # ── Training curves ───────────────────────────────────────────────────────
    epochs_range = range(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_range, history.history["loss"],     "b-o", label="Train")
    axes[0].plot(epochs_range, history.history["val_loss"], "r-o", label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history.history["accuracy"],     "b-o", label="Train")
    axes[1].plot(epochs_range, history.history["val_accuracy"], "r-o", label="Val")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

    fig.suptitle("Training Curves — DeBERTa-v3-small", fontweight="bold")
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()
    print("Training curves saved ✓")

    return acc

acc = evaluate(model, test_ds, history, cfg)


Running inference on test set …

  TEST SET RESULTS
  Accuracy : 0.9800
              precision    recall  f1-score   support

     Genuine     0.9870    0.9728    0.9799      2500
        Fake     0.9732    0.9872    0.9801      2500

    accuracy                         0.9800      5000
   macro avg     0.9801    0.9800    0.9800      5000
weighted avg     0.9801    0.9800    0.9800      5000

Confusion matrix saved ✓
Training curves saved ✓


In [7]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

model.save_pretrained(cfg.OUTPUT_DIR)
tokenizer.save_pretrained(cfg.OUTPUT_DIR)

print(f"Model saved     → {cfg.OUTPUT_DIR}")
print(f"Tokenizer saved → {cfg.OUTPUT_DIR}")

# ── List saved files ─────────────────────────────────────────────────────────
print("\nSaved files:")
for f in sorted(os.listdir(cfg.OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(cfg.OUTPUT_DIR, f))
    print(f"  {f:<35} {size/1e6:.1f} MB")

# ── Flask loading snippet ─────────────────────────────────────────────────────
print("""
Flask backend loading snippet:
──────────────────────────────
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

MODEL_DIR = "backend/model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = TFAutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
""")

print("Done ✓")

Model saved     → /kaggle/working/backend/model
Tokenizer saved → /kaggle/working/backend/model

Saved files:
  added_tokens.json                   0.0 MB
  config.json                         0.0 MB
  special_tokens_map.json             0.0 MB
  spm.model                           2.5 MB
  tf_model.h5                         567.7 MB
  tokenizer.json                      8.7 MB
  tokenizer_config.json               0.0 MB

Flask backend loading snippet:
──────────────────────────────
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

MODEL_DIR = "backend/model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = TFAutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

Done ✓


In [8]:
import shutil

# ── Zip the model directory ───────────────────────────────────────────────────
print("\nZipping model files for download...")

# The destination path for the zip file (shutil automatically appends '.zip')
zip_base_name = "/kaggle/working/deberta_v3_finetuned"

# Create the archive
shutil.make_archive(
    base_name=zip_base_name, 
    format='zip', 
    root_dir=cfg.OUTPUT_DIR
)

print(f"Zip file created successfully ✓")
print(f"File path: {zip_base_name}.zip")
print("\nYou can now refresh the Kaggle output pane (on the right) and download the single .zip file!")


Zipping model files for download...
Zip file created successfully ✓
File path: /kaggle/working/deberta_v3_finetuned.zip

You can now refresh the Kaggle output pane (on the right) and download the single .zip file!
